# Day 9 ｜ 异常处理与防御性编程

**日期**：2026-10-27 周二　**Day 9 / 84**　**第 2 周**

今天的东西你已经**用过三次**，只是不知道它叫什么：

| 你在哪见过 | 那是什么 |
|---|---|
| 昨天换目录跑工具，崩了 | `FileNotFoundError` |
| 脚手架里那些 `raise NotImplementedError` | 主动抛异常 |
| `to_number` 里那个 `try / except ValueError` | 今天的主角 |

**所以今天不是学新语法，是给已经在用的东西补上名字、规则和边界。**

## 零散时间安排

| 节 | 内容 | 用时 | 优先级 |
|---|---|---|---|
| 复习 | D-1 / D-3 / D-7 | 4分 | 必做 |
| 1 | 异常到底是什么 | 5分 | 必做 |
| 2 | try / except 基本形 | 6分 | 必做 |
| 3 | 五种常见异常速查 | 10分 | 必做（当字典用）|
| 4 | ⚠️ 裸 except 为什么危险 | 8分 | **今天最重要** |
| 5 | else / finally | 4分 | 了解 |
| 6 | 读 traceback | 8分 | **本周主线** |
| 7 | raise：失败要响 | 5分 | 必做 |
| 8 | 什么该 try、什么不该 | 6分 | 必做 |
| 练习 | A 给 clean_row 加容错 / B 投毒测试 | 20分 | A 必做 |

---
## 🔁 开场复习（4 分钟，凭记忆答）

```
D-1（Day 8 函数）   全局变量有哪两种坏法？「读全局」和「写全局」
                    分别坏在哪、分别该用什么替代？

D-3（Day 6 工具）   classify_lines 和 clean_row 分别管哪一层？
                    为什么第18行的逗号不在 classify_lines 里删掉？

D-7（Day 2 类型）   int(-3.5) 等于几？-3.5 // 1 等于几？
                    （这条你答错过三次了，今天再确认一次）

错题抽查            int(float('3.5')) 为什么引号只包 '3.5' 不包 float？
```

---

## 1. 异常到底是什么　*约 5 分钟*

一句话：**异常 = 程序遇到没法继续的情况时，主动停下来并且喊一声。**

它不是"bug"。bug 是你写错了；异常是**运行时的意外**。而且它有个关键设计：

> **异常发生的那一刻，函数立刻退出，后面所有代码都不执行。**

这个"立刻退出"既是它的价值（不会拿错误数据继续算），也是它的危险（你没接住，整个程序就没了）。

In [ ]:
def divide(a, b):
    print("  进函数了")
    result = a / b                      # ← 这里出事的话，下面全部作废
    print("  只有上面不炸，这行才会打印")
    return result

print("正常情况：", divide(10, 2))
print()
print("b=0 的情况：")
try:
    divide(10, 0)
except ZeroDivisionError as e:
    print(f"  被接住了：ZeroDivisionError: {e}")
print("  ↑ 注意第二句 print 没有执行 —— 异常让函数当场退出")
print("  但因为外面有 try，程序本身没死，继续跑完了")

**如果外面没有 try 会怎样？** 就是你昨天亲眼看到的：

```
FileNotFoundError: [Errno 2] No such file or directory: 'week01/day06_dirty.txt'
Traceback (most recent call last):
  ...
```

程序直接终止，后面的汇总一行都没打印。**这就是"一行坏数据废掉整份报告"**——Day 6 我顶回你"非法输入报错"那个需求的原因。今天给你实现那个决定的工具。

> ### ⏸ 检查点 1

---
## 2. try / except 基本形　*约 6 分钟*

```python
try:
    可能出事的那一句              # ← 只放这一句，别把整段逻辑塞进来
except 异常类型 as e:
    出事了怎么办                   # ← 必须做点什么：记下来 / 返回 None / 换默认值
```

In [ ]:
# 三种处理方式，选哪种是设计决策，不是语法问题
def to_number_v1(text):
    """方式A：失败返回 None，交给调用方决定。← 你现在的 to_number 就是这个"""
    try:
        return float(text)
    except ValueError:
        return None

def to_number_v2(text, default=0.0):
    """方式B：失败用默认值顶上。危险：错误数据会被当成合法数据算下去。"""
    try:
        return float(text)
    except ValueError:
        return default

def to_number_v3(text):
    """方式C：包一层信息再抛出去，让上层知道是哪个字段坏了。"""
    try:
        return float(text)
    except ValueError:
        raise ValueError(f'金额字段无法解析：{text!r}') from None

print("A  to_number_v1('abc') →", to_number_v1('abc'))
print("B  to_number_v2('abc') →", to_number_v2('abc'), ' ← 危险：0 会被当成真实金额加进汇总')
print("C  to_number_v3('abc') ↓")
try:
    to_number_v3('abc')
except ValueError as e:
    print(f"   ValueError: {e}   ← 这个信息能让人三分钟定位问题")
print()
print("  ★ 结论：方式A 最常用，方式C 用于需要留痕的关键处。")
print("    方式B 看着方便，但它把『坏数据』变成了『看起来正常的假数据』——")
print("    最后你的报表里会多出一堆 0，而没人知道为什么。这叫静默污染。")

A  to_number_v1('abc') → 0.1
B  to_number_v2('abc') → 0.0  ← 危险：0 会被当成真实金额加进汇总
C  to_number_v3('abc') ↓
   ValueError: 金额字段无法解析：'abc'   ← 这个信息能让人三分钟定位问题

  ★ 结论：方式A 最常用，方式C 用于需要留痕的关键处。
    方式B 看着方便，但它把『坏数据』变成了『看起来正常的假数据』——
    最后你的报表里会多出一堆 0，而没人知道为什么。这叫静默污染。


**🔮 预测**：下面两个 cell，哪个会崩？

In [3]:
data = {"品名": "玻璃", "数量": 100}

# 第一个
print("取存在的键:", data["品名"])

# 第二个会怎样？想一想再运行
try:
    print(data["单价"])
except KeyError as e:
    print(f"KeyError: {e}  ← 键不存在，直接 [] 取值就是这个下场")

print("用 get 就安全:", data.get("单价"), "（返回 None，不崩）")
print()
print("  这正好对上你 Day 5 那条：字典 [键] 会 KeyError，.get() 不会。")

取存在的键: 玻璃
KeyError: '单价'  ← 键不存在，直接 [] 取值就是这个下场
用 get 就安全: None （返回 None，不崩）

  这正好对上你 Day 5 那条：字典 [键] 会 KeyError，.get() 不会。


> ### ⏸ 检查点 2

---
## 3. 五种常见异常速查　*约 10 分钟 —— 当字典用，不用背*

**这五种占你以后会遇到的异常的 85%。** 不用背，看懂规律就行：
每一种都是"你要求的和实际拿到的不匹配"。

In [4]:
cases = []

# ── ① ValueError：类型对了，但值不对 ──
try:
    int("3.5")
except ValueError as e:
    cases.append(("ValueError", "类型对（是字符串）但内容不合法", str(e)))

# ── ② KeyError：字典里没这个键 ──
try:
    {"a": 1}["b"]
except KeyError as e:
    cases.append(("KeyError", "字典里没有这个键", str(e)))

# ── ③ IndexError：列表里没有这个位置 ──
try:
    [1, 2][5]
except IndexError as e:
    cases.append(("IndexError", "列表没有那个下标", str(e)))

# ── ④ TypeError：类型压根就不对，操作不适用 ──
try:
    "abc" + 5
except TypeError as e:
    cases.append(("TypeError", "这个类型不支持这个操作", str(e)))

# ── ⑤ AttributeError：对象上没有这个方法/属性 ──
try:
    "abc".push("d")
except AttributeError as e:
    cases.append(("AttributeError", "字符串没有 push 这个方法", str(e)))

for name, meaning, msg in cases:
    print(f"  {name:<16} {meaning}")
    print(f"  {'':16} 报错原文：{msg}")
    print()

  ValueError       类型对（是字符串）但内容不合法
                   报错原文：invalid literal for int() with base 10: '3.5'

  KeyError         字典里没有这个键
                   报错原文：'b'

  IndexError       列表没有那个下标
                   报错原文：list index out of range

  TypeError        这个类型不支持这个操作
                   报错原文：can only concatenate str (not "int") to str

  AttributeError   字符串没有 push 这个方法
                   报错原文：'str' object has no attribute 'push'



### 两张最容易混的对撞

**ValueError vs TypeError**
```
int("3.5")   → ValueError  字符串能表示数字吗？内容问题
int([])      → TypeError   列表压根不能转数字，类型问题
"abc" + 5    → TypeError   字符串加不了整数
float("abc") → ValueError  是字符串，但内容不是数
```
**记忆**：**ValueError = 类型没问题，内容不行。TypeError = 类型本身就不行。**

**KeyError vs IndexError**
```
{"a":1}[5]   → KeyError     字典按不存在的键取
[1,2][5]     → IndexError   列表按不存在的位置取
```

**还有一个 Python 初学者必踩的：**

In [5]:
parts = "a,b".split(",")
print("parts 有", len(parts), "个元素，合法下标是 0 到", len(parts) - 1)
try:
    print(parts[2])
except IndexError as e:
    print(f"  IndexError: {e}")
print()
print("  ⚠️ 为什么这对你特别重要：你的 clean_row 里会写 parts[0]...parts[6]。")
print("     万一某行只有 5 个字段（少打了逗号），parts[6] 就 IndexError。")
print("     而我们在 classify_lines 里已经用 len 检查拦掉了这种行 ——")
print("     这就是为什么『分层检查』能让后面的代码写得更简单。")
print("     你昨天那句『顺手在这里删逗号不就完了』的代价，就在这里显形。")

parts 有 2 个元素，合法下标是 0 到 1
  IndexError: list index out of range

  ⚠️ 为什么这对你特别重要：你的 clean_row 里会写 parts[0]...parts[6]。
     万一某行只有 5 个字段（少打了逗号），parts[6] 就 IndexError。
     而我们在 classify_lines 里已经用 len 检查拦掉了这种行 ——
     这就是为什么『分层检查』能让后面的代码写得更简单。
     你昨天那句『顺手在这里删逗号不就完了』的代价，就在这里显形。


> ### ⏸ 检查点 3
> 不用背五种。要背的是那条判断：**「类型不对」还是「内容不对」**，它决定你 catch 哪个。

---
## 4. ⚠️ 裸 except 为什么危险　*约 8 分钟 —— 今天最重要的一节*

你可能会想：既然异常会崩，那我 `except:` 全都接住，程序不就永远不会死了？

**这是新手能犯的最严重的错误。** 我们现场演示它怎么把一个真 bug 藏起来。

In [6]:
# ── 有 bug 的函数：把 '金额' 写成了 '钱' ──
records = [{"品名": "玻璃", "金额": 100}, {"品名": "胶膜", "金额": 50}]

def sum_amounts_buggy(data):
    total = 0
    for r in data:
        try:
            total += r["钱"]           # ← 打错了！应该是 "金额"
        except:                         # 裸 except：什么都接
            pass                        # 接住了，然后什么都不做
    return total

result = sum_amounts_buggy(records)
print(f"  裸 except 版本算出：{result}")
print(f"  正确答案应该是：150")
print()
print("  ★★★ 程序没报错、跑完了、返回了一个数字、看起来一切正常。")
print("      而它悄悄漏掉了全部数据。这种代码上线，你会在三天后")
print("      被客户问『为什么金额都是 0』的时候才发现。")
print()
print("  现在改成只 catch 该 catch 的：")
def sum_amounts_right(data):
    total = 0
    for r in data:
        total += r["钱"]               # 同样的拼写错误
    return total

try:
    sum_amounts_right(records)
except KeyError as e:
    print(f"  KeyError: {e}  ← 程序当场告诉你：键不存在")
    print("  你三秒钟就知道是键名写错了。这就是『早失败』的价值。")

  裸 except 版本算出：0
  正确答案应该是：150

  ★★★ 程序没报错、跑完了、返回了一个数字、看起来一切正常。
      而它悄悄漏掉了全部数据。这种代码上线，你会在三天后
      被客户问『为什么金额都是 0』的时候才发现。

  现在改成只 catch 该 catch 的：
  KeyError: '钱'  ← 程序当场告诉你：键不存在
  你三秒钟就知道是键名写错了。这就是『早失败』的价值。


### 三条裸 except 规矩

| 写法 | 问题 |
|---|---|
| `except:` | **最糟**。连 `KeyboardInterrupt`（你按 Ctrl+C）都接，程序可能都停不下来 |
| `except Exception:` | 比上一个好，但仍然会吞掉 `KeyError`/`TypeError` 这些**你自己的编码错误** |
| `except ValueError:` | ✅ 只接你预料到的那一种，其他照样炸出来 |

**核心逻辑：异常有两个用途 —— 处理意外，以及暴露你的错误。** 裸 except 把第二个用途杀掉了。
你最需要它响的时候，恰恰是你代码写错的时候。

**还有第二条：except 里必须做点什么。** 光 `pass` = 静默失败，比崩掉更糟。

In [7]:
# 如果确实需要"兜住一切"，至少要说出来
def process_line_dignified(raw_line, line_num):
    try:
        parts = raw_line.split(",")
        return {"行号": line_num, "字段数": len(parts)}
    except Exception as e:                    # 兜住一切时，必须留痕
        print(f"  ⚠️ 第{line_num}行处理失败：{type(e).__name__}: {e}，已跳过")
        return None

print("正常：", process_line_dignified("a,b,c", 1))
print("故意喂个奇怪的东西：")
process_line_dignified(None, 99)              # None 没有 split 方法
print("  ↑ 虽然也兜住了，但至少屏幕上留了一行，你能查到是谁、为什么")

正常： {'行号': 1, '字段数': 3}
故意喂个奇怪的东西：
  ⚠️ 第99行处理失败：AttributeError: 'NoneType' object has no attribute 'split'，已跳过
  ↑ 虽然也兜住了，但至少屏幕上留了一行，你能查到是谁、为什么


> ### ⏸ 检查点 4（重点）
> 判断标准一句话：**如果这个 except 会让某个 bug 从"立刻报错"变成"悄悄出错"，那它就是错的。**

---
## 5. else 和 finally　*约 4 分钟 —— 了解即可*

```python
try:
    做事
except ValueError:
    出事了
else:
    没出事才做               # 常用于"成功了才写库/才记账"
finally:
    无论如何都做             # 常用于关文件、清理临时资源
```

**说实话：`else` 和 `finally` 你在未来三个月里几乎用不到。** `with` 已经帮你处理了资源清理（还记得 Day 6 那个 `with open(...)` 吗？它就是 `finally` 的替身）。今天让你知道有这两个词、读到能看懂就够。

In [ ]:
def try_all(raw, expect_fields=3):
    try:
        parts = raw.split(",")
        count = len(parts)
    except AttributeError as e:
        print(f"  输入根本不是字符串：{e}")
        return
    else:                                   # 只有 try 没出事才走这里
        print(f"  切出 {count} 段，没出问题 ✓")
        if count != expect_fields:
            print(f"    但字段数不对（应 {expect_fields}）")
    finally:
        print("  收尾：无论成功失败都执行（比如记一条日志）")

try_all("a,b,c")
print()
try_all(None)

> ### ⏸ 检查点 5

---
## 6. 读 traceback　*约 8 分钟 —— 本周主线*

**这是第 2 周真正要练的技能。** 因为 AI 给你写代码，报错的时候只有你自己能读。

规则只有一条：**从最后一行往上读，不要从第一行往下读。**

- **最后一行** = 真正的错误是什么（`ValueError: could not convert string to float: 'abc'`）
- **倒数几行** = 错误发生在哪个文件的哪一行
- **往上** = 谁调用了它，一层层是调用链
- 前面的 `Traceback (most recent call last):` 是套话，直接跳过

In [ ]:
# 造一个三层调用的真实报错，看清结构
def parse_amount(text):
    return float(text)                       # ← 最深层，错误真正发生地

def clean_record(rec):
    return parse_amount(rec["金额"])          # ← 中间层

def run(records):
    return [clean_record(r) for r in records]  # ← 最外层

try:
    run([{"金额": "100"}, {"金额": "abc"}])
except Exception as e:
    import traceback
    tb = traceback.format_exc().strip().split(chr(10))
    print("真实 traceback 长这样（共", len(tb), "行）：")
    for i, line in enumerate(tb, 1):
        print(f"  {i:>2} | {line}")
    print()
    print("  读法：")
    print("    最后一行 → ValueError: could not convert string to float: 'abc'")
    print("               什么错 + 具体哪个值，答案就在这")
    print("    往上找自己写的文件行号 → parse_amount / clean_record / run")
    print("               这就是调用链：谁把 'abc' 一路传进来的")
    print("    开头的 Traceback (most recent call last) → 套话，跳过")

### 一个立刻能用的提问模板

以后卡住时,把这三样一起发我(或任何 AI),解决速度差五倍:

```
【环境】Python 3.12，项目虚拟环境，本地
【我想做】把金额字段转成 float
【实际发生】报 ValueError
【完整 traceback】（整段粘贴，不要手抄或截断 —— 截断了就看不到调用链）
【我已经试过】我以为 '1,234' 能被 float 解析，但报错了
【要求】先说原因，再给最小改动，不要重写整个文件
```

**为什么"完整 traceback"这一条不能省**：错误发生的位置常常不是原因所在。
你的 `to_number` 报错,真凶可能是十行外把列表当字符串传进来了。**只贴最后一行,信息就丢了。**

> ### ⏸ 检查点 6

---
## 7. raise：失败要响　*约 5 分钟*

你已经见过它了 —— 脚手架里那五个 `raise NotImplementedError(...)`。

**`raise` 是主动制造异常。** 用途是:**当函数发现自己拿到的东西不对,不要猜,直接喊。**

In [ ]:
def clean_row_v2(parts, line_num):
    """演示：什么时候该 raise，什么时候该返回 None。"""
    if len(parts) != 7:
        # 这是【调用方的错误】—— 传进来的东西就不该是这个形状。
        # 让它炸， programmer 才会去修，而不是悄悄产出垃圾。
        raise ValueError(
            f"clean_row 需要 7 个字段，收到 {len(parts)} 个（第{line_num}行）")

    if not parts[3].strip():
        # 这是【数据本身的问题】—— 正常业务情况，可能发生无数次。
        # 不该炸，返回 None 让上层记进坏行清单。
        return None

    return {"行号": line_num, "品名": parts[0], "数量文本": parts[3]}

print("正常：", clean_row_v2(["玻璃", "a", "b", "100", "d", "e", "f"], 2))
print("数据缺失：", clean_row_v2(["玻璃", "a", "b", "", "e", "f", "g"], 3))
print()
print("调用方传错形状：")
try:
    clean_row_v2(["只有", "三个"], 4)
except ValueError as e:
    print(f"  ValueError: {e}")

print()
print("  ★ 这条分界线是 API 设计的核心，你现在做的工具已经在用了：")
print("    程序写错了 → raise，要响，越早越好")
print("    数据不干净 → 返回 None / 记进报告，不要响，继续处理下一行")
print("    混用结果：要么用户被一堆报错淹没，要么错误数据悄悄进了报表。")

**顺带一个 `from None`：** 你大概见过我写 `raise ValueError(...) from None`。
它的作用是**不显示"During handling of the above exception..."那一段**,让报错更干净。今天知道有这东西就行。

> ### ⏸ 检查点 7

---
## 8. 什么该 try、什么不该　*约 6 分钟*

新手学会 try 之后会犯的下一个错:**什么都包起来。**

正确的原则一句话:

> **在边界处校验,在意外处捕获。**

边界 = 你和外部世界交接的地方。你的程序只有四个边界:

In [ ]:
print("该 try / 该校验的四类边界：")
print()
print("  ① 读文件      FileNotFoundError, UnicodeDecodeError")
print("     → 你的 split_lines。文件可能不存在、可能被别人移动了。")
print()
print("  ② 解析外部数据  ValueError, json.JSONDecodeError, KeyError")
print("     → 你的 to_number、以及未来的 API 返回、大模型的 JSON 输出。")
print("        ★ 这条是你第 8-10 周的重点：大模型说好返回 JSON，它会返")
print("          回带 ```json 标记的、缺逗号的、多字段的。必须兜住。")
print()
print("  ③ 网络调用    Timeout, ConnectionError, 429 限流")
print("     → 现在还没到，第 8 周调 API 时天天用。")
print()
print("  ④ 用户输入    类型错、空、超长、恶意内容")
print("     → Day 6 你写的『非法输入要报错』，其实是这一类，应该跳过+报告。")
print()
print("不该 try 的：")
print("  ✗ 自己的逻辑（if 条件、循环边界、字典键名拼错）")
print("     —— 让这些炸出来，炸得越早你改得越快")
print("  ✗ 你确定不会发生的情况（包起来纯属浪费，还会掩盖未来的改动）")
print()
print("  ★ 一个反直觉但有用的判断：")
print("     如果你写 except 的时候说不清『什么情况下会走到这里』，")
print("     那就不该写这个 except。先跑一次看看它到底会不会发生。")

### 一个能立刻用的替代方案：先校验,再操作

不是所有防御都要 try。很多时候 `if` 更清楚:

```python
# 用 try 兜
try:
    value = data["金额"]
except KeyError:
    value = 0

# 用 get —— 更短更清楚
value = data.get("金额", 0)

# 用 if —— 需要区分"不存在"和"存在但是0"时
value = data["金额"] if "金额" in data else None
```

**三种都合法,选哪个取决于你要不要区分「没这个字段」和「这个字段是 0」。**
在你的清洗工具里,**必须区分** —— 数量为 0 是业务异常,字段不存在是解析异常,两者要进不同的报告栏。

> ### ⏸ 检查点 8

---
# 🔨 今日练习

**练习 A 一箭双雕:它既是 Day 9 的容错练习,也是 Day 8 欠你的 `clean_row`。**

## 练习 A：写 `clean_row`，并且让它打不崩　*必做，约 15 分钟*

打开 `week01/day06_cleaner_v2.py`,填 `clean_row`。契约我已经定死:

```python
def clean_row(parts: list[str], line_num: int) -> tuple[dict, list]:
    record, row_repairs = clean_row(...)      # 主流程要求返回两个值
```

**分两小步做,别一口气写完:**

**A1. 先只做值清洗,不管容错**（平铺版第 2 步搬过来）


- 七个字段各自 strip
- 日期:`to_halfwidth` + 斜杠换连字符
- 品名:`normalize_key` + `NAME_ALIAS` 查表,查不到就原样保留
- 类别:只去空格,**不做归并**
- 单位:查 `UNIT_MAP`,不在表里就保留原值 + 记「单位未知」
- 数量/单价/原金额:`to_number`
- 返回 `(record, row_repairs)`

跑通的标准:`.venv/bin/python week01/day06_cleaner_v2.py` 不再报 `NotImplementedError`
（会在 `validate_and_recompute` 那里停,那是对的）。

**A2. 再回答三个问题,决定要不要加防御**（这步比写代码重要）

```
Q1  parts[6] 这一句,什么情况下会 IndexError？
    你的 classify_lines 是不是已经拦住了？如果拦住了,这里还需要 try 吗？

Q2  如果明天有人改动了 classify_lines,把 len 检查删掉了,
    clean_row 里没做防御 → 会发生什么？做了防御 → 会发生什么？
    你倾向哪种？为什么？

Q3  row_repairs 为什么必须走返回值,不能像平铺版那样
    往全局 repair_log.append？（昨天答反了的那题,今天答对了吗）
```

**把三个答案写进 NOTES.md,再决定 A2 要不要动手。**
提示:Q1 的正确答案会让大多数人省掉一个 try —— 而这正是今天第 8 节要教你的东西。

## 练习 B：投毒测试　*必做，约 8 分钟*

**这是今天的核心产出物:证明你的工具"故意搞坏也崩不了"。**

自己造一个 `week01/day06_evil.txt`,把这些垃圾行混进去:

```
（空行）
完全一行没有逗号的文字
申报日期,商品名称,类别,数量,单位,单价,金额,多一个字段
abc,def
金额,,，中文逗号
2026-10-28,测试,测试,不是数字,个,abc,xyz
2026-10-28,负数行,测试,-100,个,5,-500
```

然后跑清洗工具,**验收标准三条**:

1. 程序**不崩**,跑到最后正常打印汇总
2. 坏行进对应的报告栏,**一条都不静默消失**
3. 原本 15 行好数据的汇总结果**不受影响**(`good 15`,总额 `4,131,260.00`)

第 3 条最容易被忽略,但它是重点:**加了容错,不能改变正确输入的行为。**
（这就是重构那套验收标准的复用 —— 你已经在无意识地用它了。）

⚠️ 提醒:`FILE_PATH` 还是相对路径,换目录跑照样 `FileNotFoundError`。**那不是今天要修的**,明天 Day 10 用 pathlib 正式治它。今天你老实从项目根目录跑。

## 练习 C（选做）：故意制造四种异常

各造一次并 catch 住,巩固速查表:
`ValueError` / `KeyError` / `IndexError` / `AttributeError`。

**加一题思考**:用 `int("3,500")` 和 `int("３５００")`（全角）分别试,
哪个炸哪个不炸?猜完再跑。跑完你会对"`float()` 原生认全角数字"这件事印象深一点
—— Day 8 我们在 `to_number` 里手动做了全角转换,现在想想那一步对 float 是必要的吗?

---
## 今日收尾

1. **NOTES.md 加 Day 9 笔记**,这次多加两栏:
   - 今天验证了什么(错题集机制)
   - Q1/Q2/Q3 的答案(练习 A 的三问)
2. **错题集**:今天新增两条候选
   - 裸 except 会吞掉你自己的编码错误(⬜)
   - 「程序跑完了没报错」≠「结果正确」—— 那个漏掉全部金额的 0 就是证据(⬜)
   顺带去 #2 那条 `int()` 向零截断**标上「第9天第三次答错」**。
3. 提交:
```
cd ~/Documents/AI自学/ai-product-journey
git add .
git status
git commit -m "Day 9: 异常处理与防御性编程 + clean_row 实现"
git push
```

**明天 Day 10 是文件与数据格式**:`with open` 正式讲、pathlib 修掉那个相对路径的坑、JSON/CSV 对比。你今天的 `FileNotFoundError` 和明天正好接上。